In [1]:
## importing the important libraries:
import os
from dotenv import load_dotenv
load_dotenv()

from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence.aio import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import DocumentAnalysisFeature
from azure.ai.documentintelligence.models import DocumentContentFormat
from openai import AsyncAzureOpenAI
import base64


In [2]:
file_path = r"C:\Users\SumeetMaheshwari\Downloads\Input 1_pdf.pdf"

# Load environment variables
endpoint = os.getenv("DOCUMENT_INTELLIGENCE_ENDPOINT")
key = os.getenv("DOCUMENT_INTELLIGENCE_KEY")

In [3]:
### function for the reading the pdf as input and return the result
document_intelligence_client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

async def data_extractor(pdf_path:str):
    with open(pdf_path, "rb") as f:
        pdf_bytes = f.read()

    # Base64 encode PDF
    base64_encoded_pdf = base64.b64encode(pdf_bytes).decode("utf-8")

    analyze_request = {
        "base64Source": base64_encoded_pdf
    }

    # Start analysis
    poller = await document_intelligence_client.begin_analyze_document(
        "prebuilt-layout",
        analyze_request,
        output_content_format=DocumentContentFormat.MARKDOWN
        
    )
    
    result = await poller.result()
    page_wise_md = result.content.split("<!-- PageBreak -->")

    page_wise_ocr = []

    for page_idx, page in enumerate(result.pages):
        page_wise_context = ""
        if not(page.lines):
            import pdb;pdb.set_trace()
        for line_idx, line in enumerate(page.lines):
            page_wise_context += line.content

        page_wise_ocr.append(page_wise_context)

    return page_wise_md, page_wise_ocr


In [4]:
page_wise_md, page_wise_ocr = await data_extractor(file_path)

# Text Cleaning:

In [26]:
for i in range(len(page_wise_ocr)):
    print(page_wise_ocr[i])

Product Name:Aripiprazole Tablets USP 5 mgAlembicTouching Lives over100yearsBMR No. & VersionNo.F1\BMR\00837 & 3.0Product Code:30000773Batch Size in Kg /Liter:142.500 kgBatch Size inUnit:1,500,000 TabletsTable of ContentsSr. No.Process StagePage No.1Table of Contents12Batch Information Sheet23Abbreviations34Safety Instruction45Manufacturing Process5-556Batch History card56-577Signature Log58-598Change History of Document60Format No .: C\QASOP\0107-F001-1.0Effective Date: 09/07/2024
Product Name:Aripiprazole Tablets USP 5 mgAlembicTouching Lives over 1100yearsBMR No. & VersionNo.F1\BMR\00837 & 3.0Product Code:30000773Batch Size in Kg /Liter:142.500 kgBatch Size inUnit:1,500,000 TabletsBATCH MANUFACTURING RECORDBatch Information Sheet01Generic Name of ProductAripiprazole Tablets USP02Brand Name of ProductNA03Label ClaimEach tablet contains 5 mg of Aripiprazole USP.04Storage ConditionStore in tightly closed containers at 25°℃ (77ºF);excursions permitted to 15°-30℃ (59º-86ºF) [see USPContr

In [37]:
import re

def remove_header_footer_flexible(text):
    """
    Removes headers/footers without inserting artificial spaces.
    Prevents word merging while preserving original formatting.
    """

    header_patterns = [
        r'Product\s*Name:\s*Aripiprazole\s*Tablets\s*USP\s*5\s*mg.*?Touching\s*Lives\s*over\s*\d+\s*years',
        r'BMR\s*No\.\s*&\s*Version\s*No\.\s*F1\\BMR\\\d+\s*&\s*\d+\.\d+\s*Product\s*Code:\s*\d+',
        r'Batch\s*Size\s*in\s*Kg\s*/\s*Liter:\s*[\d,\.]+\s*kg\s*Batch\s*Size\s*in\s*Unit:\s*[\d,\.]+\s*Tablets',
        r'Product\s*Name:.*?Tablets',
        r'Format\s*No\.?\s*:\s*C\\QA\\SOP\\\d+-F\d+-\d+\.\d+\s*Effective\s*Date\s*:\s*\d{2}/\d{2}/\d{4}',
        r'Alembic\s*Touching\s*Lives\s*over\s*\d+\s*years',
    ]

    footer_patterns = [
        r'Format\s*No\.?\s*:\s*C\\QA\\SOP\\[A-Z0-9\-\.]+\s*Effective\s*Date\s*:\s*\d{2}/\d{2}/\d{4}',
        r'Format\s*No\.?\s*:\s*C\\QASOP\\[A-Z0-9\-\.]+\s*Effective\s*Date\s*:\s*\d{2}/\d{2}/\d{4}',
        r':\s*Format\s*No\.?\s*:\s*C\\QASOP\\[A-Z0-9\-\.]+\s*Effective\s*Date\s*:\s*\d{2}/\d{2}/\d{4}',
        r'No\.?\s*Name\s*Signature.*?Format\s*No\.?\s*:\s*C\\QA\\SOP\\[A-Z0-9\-\.]+\s*Effective\s*Date\s*:\s*\d{2}/\d{2}/\d{4}',
    ]

    cleaned_text = text

    # Remove headers
    for pattern in header_patterns:
        cleaned_text = re.sub(
            pattern,
            '\n',
            cleaned_text,
            flags=re.IGNORECASE | re.DOTALL
        )

    # Remove footers
    for pattern in footer_patterns:
        cleaned_text = re.sub(
            pattern,
            '\n',
            cleaned_text,
            flags=re.IGNORECASE | re.DOTALL
        )

    # Normalize newlines only (DO NOT force spaces)
    cleaned_text = re.sub(r'\n{3,}', '\n\n', cleaned_text)
    cleaned_text = cleaned_text.strip()

    return cleaned_text


In [38]:
cleaned_text_ocr = [
    remove_header_footer_flexible(text)
    for text in page_wise_ocr
]


In [39]:
len(cleaned_text_ocr)

60

In [40]:
cleaned_text = "\n\n".join(cleaned_text_ocr)

In [41]:
print(cleaned_text)

Table of ContentsSr. No.Process StagePage No.1Table of Contents12Batch Information Sheet23Abbreviations34Safety Instruction45Manufacturing Process5-556Batch History card56-577Signature Log58-598Change History of Document60Format No .: C\QASOP\0107-F001-1.0Effective Date: 09/07/2024

BATCH MANUFACTURING RECORDBatch Information Sheet01Generic Name of ProductAripiprazole Tablets USP02Brand Name of ProductNA03Label ClaimEach tablet contains 5 mg of Aripiprazole USP.04Storage ConditionStore in tightly closed containers at 25°℃ (77ºF);excursions permitted to 15°-30℃ (59º-86ºF) [see USPControlled Room Temperature].05Stage /Dosage FormUncoated Tablet06Market/CustomerExport /US07Reference Document No.MFC/0353-0508Manufacturing license No.G/95909Manufactured ByAlembic Pharmaceuticals Limited, Formulation Unit,Village Panelav, Near Baska, Tal. Halol, Dist.Panchmahal, Gujarat.10Customer Batch No.11Issued By ( Sign/Date)12Received By (Sign/Date)13Manufacturing Date14Date of Commencement15Date of Co